In [22]:
import torch

pt = torch.load('/home/scpark/logpx_samplers/samplings/DiT/1.5/200/Euler/1000/train_traj_4/0.pt')
pt.keys()

dict_keys(['cond', 'sample', 'noise', 'traj', 'timesteps'])

In [24]:
pt['timesteps']

tensor([1.0000, 0.9950, 0.9900, 0.9850, 0.9800, 0.9750, 0.9700, 0.9650, 0.9600,
        0.9550, 0.9500, 0.9451, 0.9401, 0.9351, 0.9301, 0.9251, 0.9201, 0.9151,
        0.9101, 0.9051, 0.9001, 0.8951, 0.8901, 0.8851, 0.8801, 0.8751, 0.8701,
        0.8651, 0.8601, 0.8551, 0.8501, 0.8452, 0.8402, 0.8352, 0.8302, 0.8252,
        0.8202, 0.8152, 0.8102, 0.8052, 0.8002, 0.7952, 0.7902, 0.7852, 0.7802,
        0.7752, 0.7702, 0.7652, 0.7602, 0.7552, 0.7502, 0.7453, 0.7403, 0.7353,
        0.7303, 0.7253, 0.7203, 0.7153, 0.7103, 0.7053, 0.7003, 0.6953, 0.6903,
        0.6853, 0.6803, 0.6753, 0.6703, 0.6653, 0.6603, 0.6553, 0.6503, 0.6454,
        0.6404, 0.6354, 0.6304, 0.6254, 0.6204, 0.6154, 0.6104, 0.6054, 0.6004,
        0.5954, 0.5904, 0.5854, 0.5804, 0.5754, 0.5704, 0.5654, 0.5604, 0.5554,
        0.5504, 0.5455, 0.5405, 0.5355, 0.5305, 0.5255, 0.5205, 0.5155, 0.5105,
        0.5055, 0.5005, 0.4955, 0.4905, 0.4855, 0.4805, 0.4755, 0.4705, 0.4655,
        0.4605, 0.4555, 0.4506, 0.4456, 

In [38]:
def interp_traj(X, t, s, eps=1e-12):
    t, idx = t.to(X.device).sort()
    X = X[:, idx]
    s = s.to(X.device).clamp(t[0], t[-1])
    i1 = torch.bucketize(s, t).clamp(1, t.numel()-1); i0 = i1 - 1
    w  = ((s - t[i0]) / (t[i1] - t[i0] + eps)).to(X.dtype).view(1, -1, 1, 1, 1)
    return torch.lerp(X[:, i0], X[:, i1], w)


In [40]:

teacher_traj = pt['traj'].unsqueeze(0)
print(teacher_traj.shape)
teacher_timesteps = pt['timesteps']
print(teacher_timesteps.shape)
student_timesteps = torch.tensor([1, 0.77, 0.44, 0.33, 0.0])
student_traj = interp_traj(teacher_traj, teacher_timesteps, student_timesteps)
print(student_traj.shape)

torch.Size([1, 201, 4, 32, 32])
torch.Size([201])
torch.Size([1, 5, 4, 32, 32])


In [41]:
import torch
from math import isclose

# 방금 만든 interp_traj(X,t,s) 사용

# 0) 장치/형 맞추기용 유틸
dev, dt = 'cuda' if torch.cuda.is_available() else 'cpu', torch.float32

B,L,C,H,W = 2, 7, 3, 4, 5
t = torch.linspace(1, 0, L, device=dev)                 # teacher 시점 (내림차순)
X = torch.randn(B, L, C, H, W, device=dev, dtype=dt)

# 1) identity: s == 정렬된 t  →  그대로 복원
s = t.sort().values
Y = interp_traj(X, t, s)
assert torch.allclose(Y, X[:, torch.argsort(t)], atol=0, rtol=0)

# 2) 경계: s가 범위 밖이면 clamp → 양 끝과 동일
s = torch.tensor([10.0, -10.0], device=dev)
Y = interp_traj(X, t, s)
assert torch.allclose(Y[:, 0], X[:, t.argmax()], atol=0, rtol=0)  # s>max → 첫 프레임
assert torch.allclose(Y[:, 1], X[:, t.argmin()], atol=0, rtol=0)  # s<min → 끝 프레임

# 3) 선형성 테스트: 시간 자체를 값으로 가진 선형 시그널이면 오차=0
X_lin = t.view(1, L, 1, 1, 1).expand(B, L, C, H, W).to(device=dev, dtype=dt)
s = torch.tensor([1.0, 0.75, 0.5, 0.25, 0.0], device=dev)
Y = interp_traj(X_lin, t, s)
target = s.view(1, -1, 1, 1, 1).expand_as(Y)
assert torch.allclose(Y, target, atol=1e-6)

# 4) 무작위 샘플에서 모양/범위만 체크
s = torch.rand(11, device=dev) * (t.max() - t.min()) + t.min()
Y = interp_traj(X, t, s)  # (B, M, C, H, W)
assert Y.shape == (B, s.numel(), C, H, W)
print("✅ interp_traj: all sanity checks passed")


✅ interp_traj: all sanity checks passed
